## Project 2: AdventureWorks Data Lakehouse with PySpark Structured Streaming
This capstone project builds a **Medallion Architecture Data Lakehouse** for the AdventureWorks dataset using PySpark Structured Streaming. Data is ingested from three heterogeneous sources: a **MySQL** relational database, a **MongoDB Atlas** NoSQL cluster, and a **flat-file** CSV. It is then processed through Bronze, Silver, and Gold streaming layers before being persisted as managed Spark tables. This is a continuation of Project 1 combined with code from Lab06 and assistance from Gemini 3.1 Pro for debugging.

**Architecture Overview:**
- **Bronze Layer:** Raw JSON data ingested as-is into Parquet format
- **Silver Layer:** Cleaned, typed, and joined with dimension tables
- **Gold Layer:** Aggregated business-ready analytics table

## Section I: Prerequisites

### 1.0. Import Required Libraries

In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd
import numpy as np

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W
from pyspark.sql.functions import approx_count_distinct, sum, desc
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

C:\spark-4.0.2-bin-hadoop3


### 2.0. Instantiate Global Variables

In [2]:
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "adventureworks",
    "conn_props" : {
        "user" : "root",
        "password" : "Rf122137182",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

mongodb_args = {
    "cluster_location" : "atlas", 
    "user_name" : "tyd4gf",
    "password" : "Rf122137182",
    "cluster_name" : "lab3",
    "cluster_subnet" : "2g1tl9k",
    "db_name" : "adventureworks",
    "collection" : "",
    "null_column_threshold" : 0.5
}

base_dir = os.getcwd()
batch_dir = base_dir
data_dir = os.path.join(base_dir, 'adventureworks')
stream_dir = os.path.join(data_dir, 'streaming')

sales_stream_dir = os.path.join(stream_dir, 'sales_orders')

dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

sales_output_bronze = os.path.join(database_dir, 'fact_sales', 'bronze')
sales_output_silver = os.path.join(database_dir, 'fact_sales', 'silver')
sales_output_gold = os.path.join(database_dir, 'fact_sales', 'gold')


### 3.0. Define Global Functions

In [3]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))
    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    return pd.DataFrame(data=data, columns=column_names)

def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
    print(f"The stream has processed {len(query.recentProgress)} batchs")

def remove_directory_tree(path: str):
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
    except Exception as e:
        return f"An error occurred: {e}"

def drop_null_columns(df, threshold):
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    return df_dropped

def get_mysql_dataframe(spark_session, sql_query : str, **args):
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    dframe = spark_session.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", args['conn_props']['driver']) \
    .option("user", args['conn_props']['user']) \
    .option("password", args['conn_props']['password']) \
    .option("query", sql_query) \
    .load()
    return dframe

def get_mongo_uri(**args):
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"
    return uri

def get_spark_conf_args(spark_jars : list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    sparkConf_args = {
        "app_name" : "PySpark AdventureWorks Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[0:-2],
        "database_dir" : sql_warehouse_dir
    }
    return sparkConf_args

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
    .setMaster(args['worker_threads']) \
    .set('spark.driver.memory', '4g') \
    .set('spark.executor.memory', '2g') \
    .set('spark.jars', args['spark_jars']) \
    .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:10.4.0') \
    .set('spark.mongodb.read.connection.uri', args['mongo_uri']) \
    .set('spark.mongodb.write.connection.uri', args['mongo_uri']) \
    .set('spark.sql.adaptive.enabled', 'false') \
    .set('spark.sql.debug.maxToStringFields', 35) \
    .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
    .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .set('spark.sql.streaming.schemaInference', 'true') \
    .set('spark.sql.warehouse.dir', args['database_dir']) \
    .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    return sparkConf

def get_mongo_client(**args):
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())
    elif args['cluster_location'] == "local":
        client = pymongo.MongoClient(mongo_uri)
    else:
        raise Exception("A MongoDB Client could not be created.")
    return client

def get_mongodb_dataframe(spark_session, **args): 
    client = get_mongo_client(**args)
    db = client[args['db_name']]
    pdf = pd.DataFrame(list(db[args['collection']].find({})))
    if '_id' in pdf.columns:
        pdf.drop(['_id'], axis=1, inplace=True)
    client.close()
    temp_file = f"temp_{args['collection']}.json"
    pdf.to_json(temp_file, orient='records', lines=True)
    dframe = spark_session.read.json(temp_file)
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    return dframe

### 4.0. Initialize Data Lakehouse Directory Structure
Remove the Data Lakehouse Database Directory Structure to Ensure Idempotency

In [4]:
remove_directory_tree(database_dir)

jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")
jars.append(mysql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)
sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")

### 5.0. Create a New Spark Session

In [5]:
jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")
mssql_spark_jar = os.path.join(os.getcwd(), "sqljdbc_12.8", "enu", "jars", "mssql-jdbc-12.8.1.jre11.jar")

jars.append(mysql_spark_jar)
#jars.append(mssql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)

sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

### 6.0. Create a New Metadata Database

In [6]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")
sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Project 2 Database'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'Capstone Data Lakehouse');
"""
spark.sql(sql_create_db)

DataFrame[]

## Section II: Populate Dimensions by Ingesting "Cold-Path" Reference Data
Dimension tables are loaded from three source systems: the file system (CSV), MongoDB Atlas (JSON), and MySQL (relational). Surrogate keys are generated using PySpark SQL window functions to conform to Medallion Architecture best practices.

### 1.0. Fetch Date Data from the File System (CSV)
Reads the `dim_date.csv` flat file directly from the batch directory and saves it as a managed Spark table.

In [7]:
date_csv = os.path.join(batch_dir, 'dim_date.csv') 

df_dim_date = spark.read.format('csv').options(header='true', inferSchema='true').load(date_csv)

df_dim_date = df_dim_date.withColumnRenamed("DateKey", "date_key")

# Save to Data Lakehouse
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

# Unit Test
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

,date_key,FullDate,Year,Month,DayOfWeek
0,20010101,2001-01-01,2001,1,Monday
1,20010102,2001-01-02,2001,1,Tuesday


### 2.0. Fetch Customer Data from MongoDB Atlas
Queries the `customers` collection in the MongoDB Atlas cluster using the `get_mongodb_dataframe()` helper function, assigns a surrogate key via `ROW_NUMBER()`, and saves as `dim_customers`.

In [8]:
mongodb_args["collection"] = "customers"

df_dim_customers = get_mongodb_dataframe(spark, **mongodb_args)

df_dim_customers = df_dim_customers.withColumnRenamed("CustomerID", "customer_id")

# Generate a surrogate key (customer_key) using PySpark SQL Windowing
df_dim_customers.createOrReplaceTempView("customers")
sql_customers = """
    SELECT *, ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key
    FROM customers;
"""
df_dim_customers = spark.sql(sql_customers)

ordered_cust_columns = ['customer_key', 'customer_id', 'City', 'State_Province', 'Country_Region', 'PostalCode']
df_dim_customers = df_dim_customers[ordered_cust_columns]

# Save to Data Lakehouse
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

# Unit Test
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 2").toPandas()

,customer_key,customer_id,City,State_Province,Country_Region,PostalCode
0,1,1,Seattle,Washington,United States,98104
1,2,2,Renton,Washington,United States,98055


### 3.0. Fetch Product Data from MySQL
Reads product records from the `dim_products_vw` view in the AdventureWorks MySQL database via JDBC, selects the core product columns, and saves as `dim_products`.

In [9]:
# Extract from MySQL and generate Surrogate Key
sql_products = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY ProductID) AS product_key
    FROM {mysql_args['db_name']}.dim_products_vw
"""
df_dim_products = get_mysql_dataframe(spark, sql_products, **mysql_args)

df_dim_products = df_dim_products.withColumnRenamed("ProductID", "product_id")

ordered_prod_columns = ['product_key', 'product_id', 'Name', 'ProductNumber', 'StandardCost', 'ListPrice']
df_dim_products = df_dim_products[ordered_prod_columns]

# Save to Data Lakehouse
df_dim_products.write.saveAsTable(f"{dest_database}.dim_products", mode="overwrite")

# Unit Test
spark.sql(f"SELECT * FROM {dest_database}.dim_products LIMIT 2").toPandas()

,product_key,product_id,Name,ProductNumber,StandardCost,ListPrice
0,1,1,Adjustable Race,AR-5381,0.0,0.0
1,2,2,Bearing Ball,BA-8327,0.0,0.0


### 3.5. Create Order Status Dimension
Builds the `dim_order_status` table entirely in Spark SQL to represent all six possible AdventureWorks order statuses. This satisfies the requirement for a third supplementary dimension.

In [10]:
sql_status = f"""
    SELECT 1 AS status_id, 'In Process' AS status_name UNION ALL
    SELECT 2, 'Approved' UNION ALL
    SELECT 3, 'Backordered' UNION ALL
    SELECT 4, 'Rejected' UNION ALL
    SELECT 5, 'Shipped' UNION ALL
    SELECT 6, 'Cancelled'
"""

df_dim_status = spark.sql(sql_status)

# Save to Data Lakehouse
df_dim_status.write.saveAsTable(f"{dest_database}.dim_order_status", mode="overwrite")

# Unit Test
display(spark.sql(f"SELECT * FROM {dest_database}.dim_order_status").toPandas())

,status_id,status_name
0,3,Backordered
1,1,In Process
2,6,Cancelled
3,2,Approved
4,4,Rejected
5,5,Shipped


### 4.0. Verify Dimension Tables
Confirms that all four dimension tables have been successfully registered in the `adventureworks_dlh` metadata database.

In [11]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,dim_customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,dim_order_status,False
3,adventureworks_dlh,dim_products,False
4,,customers,True


## Section III: Process Streaming Fact Data (Medallion Architecture)
The sales fact data is segmented into three JSON files and ingested through a three-tier streaming pipeline: **Bronze** (raw ingest) → **Silver** (cleansed & joined) → **Gold** (aggregated analytics).

### 5.0. Setup: Segment Fact Data into JSON Files for Streaming
Extracts the complete sales fact table from MySQL, splits it into three equal JSON files, and writes them to the streaming source directory. If the files already exist, this step is skipped.

In [12]:
os.makedirs(sales_stream_dir, exist_ok=True)

# Check if the 3 JSON files already exist to save time
existing_files = [f for f in os.listdir(sales_stream_dir) if f.endswith('.json')]

if len(existing_files) == 3:
    print(f"Skipping extraction: Found {len(existing_files)} JSON files already in {sales_stream_dir}")
    display(get_file_info(sales_stream_dir))
    
else:
    sql_fact_sales = "SELECT * FROM fact_sales_orders_vw"
    df_fact_sales_raw = get_mysql_dataframe(spark, sql_fact_sales, **mysql_args)

    pdf_fact_sales = df_fact_sales_raw.toPandas()

    # Format Datetime columns to string for clean JSON serialization
    date_cols = ['OrderDate', 'DueDate', 'ShipDate']
    for col in date_cols:
        if col in pdf_fact_sales.columns:
            pdf_fact_sales[col] = pdf_fact_sales[col].astype(str)

    chunks = np.array_split(pdf_fact_sales, 3)

    # Export as separate JSON files to simulate incoming streams
    for i, chunk in enumerate(chunks):
        file_path = os.path.join(sales_stream_dir, f"sales_orders_0{i+1}.json")
        chunk.to_json(file_path, orient='records', lines=True)

    print(f"Successfully generated {len(chunks)} streaming JSON files in {sales_stream_dir}")
    display(get_file_info(sales_stream_dir))

Skipping extraction: Found 3 JSON files already in C:\Users\Ryan Fang\Downloads\uva stuff\yr2 semester 2\ds 2002 data systems\DS-2002\Projects\adventureworks\streaming\sales_orders


,name,size,modification_time
0,sales_orders_01.json,33299837,2026-05-08 08:26:12.506707190
1,sales_orders_02.json,33236369,2026-05-08 08:26:12.843317509
2,sales_orders_03.json,33223948,2026-05-08 08:26:13.190377235


#### Bronze Layer: Raw Data Ingest

##### 5.1.1. Read Raw JSON Files into a Stream
Configures a PySpark `readStream` to consume JSON files from the streaming source directory one file per trigger.

In [13]:
df_sales_bronze = (
    spark.readStream \
    .option("schemaLocation", sales_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .json(sales_stream_dir)
)

df_sales_bronze.isStreaming

True

##### 5.1.2. Write Streaming Data to Parquet (Bronze)
Appends `receipt_time` and `source_file` traceability columns, then writes the raw stream to Parquet in the Bronze output path.

In [14]:
sales_checkpoint_bronze = os.path.join(sales_output_bronze, '_checkpoint')

sales_bronze_query = (
    df_sales_bronze
    # Add Current Timestamp and Input Filename columns for Traceability
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("sales_bronze") \
    .trigger(availableNow = True) \
    .option("checkpointLocation", sales_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(sales_output_bronze)
)

##### 5.1.3. Unit Test: Bronze Query Monitoring
Prints the query ID, name, and status, then awaits termination to ensure all available batches have been processed before proceeding.

In [15]:
print(f"Query ID: {sales_bronze_query.id}")
print(f"Query Name: {sales_bronze_query.name}")
print(f"Query Status: {sales_bronze_query.status}")

# Await termination to ensure the stream finishes processing the available batch
sales_bronze_query.awaitTermination()

Query ID: f542d47b-feb6-4e56-8c9e-601554e0af90
Query Name: sales_bronze
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


#### Silver Layer: Cleaning Data

##### 5.2.1 & 5.2.2. Load Dimensions and Define Silver Stream Query
Reads all four dimension tables as static DataFrames, then joins them with the Bronze stream. Applies type casting and column selection to produce a clean, enriched Silver dataset.

In [16]:
# Read ALL managed dimension tables 
df_dim_cust_lh = spark.read.table(f"{dest_database}.dim_customers")
df_dim_prod_lh = spark.read.table(f"{dest_database}.dim_products")
df_dim_date_lh = spark.read.table(f"{dest_database}.dim_date")
df_dim_status_lh = spark.read.table(f"{dest_database}.dim_order_status") # <-- NEW

# Read the Bronze streaming data
df_sales_bronze_stream = spark.readStream.format("parquet").load(sales_output_bronze)

# Join the Stream with the Batch Dimensions and Clean Data Types
df_sales_silver = df_sales_bronze_stream \
    .join(df_dim_cust_lh, col("CustomerID") == col("customer_id"), "inner") \
    .join(df_dim_prod_lh, col("ProductID") == col("product_id"), "inner") \
    .join(df_dim_date_lh, col("OrderDate").cast(DateType()) == col("FullDate").cast(DateType()), "inner") \
    .join(df_dim_status_lh, col("Status").cast(IntegerType()) == col("status_id"), "left_outer") \
    .select(
        col("SalesOrderID").cast(LongType()),
        col("customer_key").cast(LongType()),
        col("product_key").cast(LongType()),
        col("date_key").cast(LongType()),
        col("status_name").alias("OrderStatus"),
        col("OrderQty").cast(IntegerType()),
        regexp_replace(col("UnitPrice"), r"[\$,]", "").cast(DoubleType()).alias("UnitPrice"),
        regexp_replace(col("LineTotal"), r"[\$,]", "").cast(DoubleType()).alias("LineTotal"),
        regexp_replace(col("TaxAmt"), r"[\$,]", "").cast(DoubleType()).alias("TaxAmt"),
        regexp_replace(col("Freight"), r"[\$,]", "").cast(DoubleType()).alias("Freight"),
        col("receipt_time"),
        col("source_file")
    )

df_sales_silver.isStreaming

True

##### Silver Schema Verification
Prints the schema of the Silver DataFrame to confirm correct column types after transformation and join.

In [17]:
df_sales_silver.printSchema()

root
 |-- SalesOrderID: long (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- date_key: long (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- LineTotal: double (nullable = true)
 |-- TaxAmt: double (nullable = true)
 |-- Freight: double (nullable = true)
 |-- receipt_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



##### 5.2.3. Write Transformed Stream to the Data Lakehouse
Writes the Silver stream in append mode to a managed Parquet table in the Data Lakehouse.

In [18]:
sales_checkpoint_silver = os.path.join(sales_output_silver, '_checkpoint')

sales_silver_query = (
    df_sales_silver.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("sales_silver") \
    .trigger(availableNow = True) \
    .option("checkpointLocation", sales_checkpoint_silver) \
    .option("compression", "snappy") \
    .start(sales_output_silver)
)

##### 5.2.4. Unit Test: Silver Query Monitoring
Prints the Silver stream's query metadata and awaits termination to confirm all micro-batches have completed.

In [19]:
print(f"Query ID: {sales_silver_query.id}")
print(f"Query Name: {sales_silver_query.name}")
print(f"Query Status: {sales_silver_query.status}")

# Await termination to safely finish processing the micro-batch
sales_silver_query.awaitTermination()

Query ID: b5d1e591-c46a-43e9-9bff-5878cbdf3058
Query Name: sales_silver
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


#### Gold Layer: Aggregated Analytics

##### 5.3.1. Read Silver Stream & Define Gold Aggregation
Reads the Silver Parquet data as a stream and joins with Customer and Date dimensions, then groups by City and Year to compute total orders and total revenue.

In [20]:
# Read Silver as a STREAM, not a batch
df_sales_gold_stream = spark.readStream.format("parquet").load(sales_output_silver) \
    .join(df_dim_cust_lh, "customer_key") \
    .join(df_dim_date_lh, "date_key") \
    .groupBy("City", "Year") \
    .agg(
        approx_count_distinct("SalesOrderID", 0.01).alias("TotalOrders"),
        sum("LineTotal").alias("TotalRevenue")
    ) \
    .orderBy(desc("TotalRevenue"))

df_sales_gold_stream.isStreaming

True

##### 5.3.2. Write Gold Stream to In-Memory Sink
Writes the aggregated Gold stream to a named in-memory table (`fact_sales_by_city_year`) using `complete` output mode.

In [21]:
sales_gold_query = (
    df_sales_gold_stream.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("fact_sales_by_city_year") \
    .start()
)

##### 5.3.3. Await Stream Processing
Waits until at least one batch has been processed before querying the in-memory sink.

In [22]:
wait_until_stream_is_ready(sales_gold_query, 1)

The stream has processed 1 batchs


##### 5.3.4. Save Final Gold Table & Display Results
Queries the in-memory streaming sink, persists the result as a managed Spark table (`gold_revenue_by_city_year`), and displays the final revenue-by-city-year analytics table.

In [23]:
# Query the in-memory streaming sink
df_fact_sales_gold_final = spark.sql("SELECT * FROM fact_sales_by_city_year")

pd.options.display.float_format = '${:,.2f}'.format

# Save as managed table and display
df_fact_sales_gold_final.write.saveAsTable(f"{dest_database}.gold_revenue_by_city_year", mode="overwrite")
spark.sql(f"SELECT * FROM {dest_database}.gold_revenue_by_city_year").toPandas()

,City,Year,TotalOrders,TotalRevenue
0,Toronto,2003,60,"$1,620,467.96"
1,Toronto,2002,57,"$1,592,322.59"
2,London,2003,326,"$1,292,292.31"
3,Paris,2003,249,"$972,535.03"
4,London,2004,338,"$803,348.38"
...,...,...,...,...
1767,Citrus Heights,2002,1,$14.13
1768,Tucson,2001,1,$11.40
1769,Chehalis,2004,1,$4.99
1770,Byron,2004,1,$4.99


## Section IV: Cleanup

### 9.0. Stop the Spark Session


In [24]:
spark.stop()